In [ ]:
# [목적] AI가 미리 정한 선택지 중 하나만 답하도록 필요한 도구와 환경을 준비합니다.
# Enum은 선택 가능한 값을 제한하고, EnumOutputParser는 AI 답변을 그 선택지로 변환합니다.
from enum import Enum
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

In [ ]:
# [목적] AI가 선택할 수 있는 색상 값을 빨강·초록·파랑으로 제한합니다.
# 이렇게 정해 두면 AI가 목록에 없는 값을 답했을 때 쉽게 확인할 수 있습니다.
class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

In [ ]:
# [목적] Colors.RED가 Enum에서 하나의 선택지로 어떻게 표현되는지 확인합니다.
# 실제 문자열 값이 필요하면 마지막 셀처럼 .value를 사용합니다.
Colors.RED

In [ ]:
# [목적] AI 응답을 Colors 목록의 값으로 읽고 검증할 파서를 만듭니다.
# get_format_instructions()는 AI에게 허용된 색상 목록을 알려주는 안내문입니다.
parser = EnumOutputParser(enum=Colors)
parser.get_format_instructions()

In [ ]:
# [목적] 물체의 색을 묻고, 허용된 색상 중 하나로 답하게 하는 프롬프트를 만듭니다.
# prompt | ChatOpenAI() | parser는 질문 작성 → AI 답변 → Enum 변환 순서로 실행합니다.
prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?

Object: {object}

Instructions: {instructions}"""
).partial(instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI() | parser

In [17]:
# [목적] '하늘'의 색을 질문해 Colors Enum 형태의 결과를 받습니다.
# invoke는 체인을 한 번 실행하고 변환이 끝난 최종 결과를 반환합니다.
response = chain.invoke({"object": "하늘"})
print(response)

Colors.BLUE


In [18]:
# [목적] response가 일반 문자열이 아니라 Colors Enum 객체인지 확인합니다.
# 자료형을 확인하면 이후 어떤 방식으로 값을 꺼내야 할지 알 수 있습니다.
type(response)

<enum 'Colors'>

In [19]:
# [목적] Colors Enum 객체에서 실제 색상 문자열만 꺼내 확인합니다.
# .value를 쓰면 Colors.BLUE 대신 '파란색'처럼 원래 저장한 값을 얻습니다.
response.value

'파란색'